# 02 — Evaluation Metrics Deep-Dive

Compares **baseline** vs **improved** configs on the same input, same simulator seed, same error rate. The only difference is the config knobs:

| | baseline | improved |
|---|---|---|
| `coercion.enabled` | False | True |
| `prompt.strictness` | minimal | strict |
| `prompt.schema_reminders` | False | True |

Because the simulator is deterministic, raw model outputs are identical between the two runs. The config knobs change how much of that noisy output the **transform** stage can recover — that's where the lift shows up.

In [1]:
import json
import sys
from dataclasses import asdict
from pathlib import Path

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "src" / "llm_etl_eval").is_dir():
    sys.path.insert(0, str(REPO_ROOT / "src"))
elif (REPO_ROOT.parent / "src" / "llm_etl_eval").is_dir():
    REPO_ROOT = REPO_ROOT.parent
    sys.path.insert(0, str(REPO_ROOT / "src"))

DATA_RAW = REPO_ROOT / "data" / "raw"
SCHEMA_PATH = REPO_ROOT / "data" / "expected" / "target_schema.json"
GOLD_PATH = REPO_ROOT / "data" / "gold" / "labels.jsonl"
BASELINE_CONFIG = REPO_ROOT / "configs" / "baseline.yaml"
IMPROVED_CONFIG = REPO_ROOT / "configs" / "improved.yaml"
WORK = REPO_ROOT / "notebooks" / "_work" / "02_eval"
WORK.mkdir(parents=True, exist_ok=True)

REPO_ROOT

WindowsPath('C:/Users/diogo/work_code/llm-etl-and-evaluation')

## Run both configs end-to-end

Helper that executes the full pipeline for one config and returns its metrics.

In [2]:
from llm_etl_eval.etl.run import run_pipeline
from llm_etl_eval.eval.metrics import compute_metrics
from llm_etl_eval.prompting.run import run_prompting
from llm_etl_eval.sim.llm_simulator import (
    SimulationConfig, load_inputs, simulate_records, write_outputs,
)
from llm_etl_eval.transform.run import run_transform


def run_config(label, config_path):
    base = WORK / label
    base.mkdir(parents=True, exist_ok=True)

    processed_dir = base / "processed"
    run_pipeline(DATA_RAW, processed_dir)
    inputs_path = processed_dir / "llm_inputs.jsonl"

    prompts_path = base / "prompts.jsonl"
    run_prompting(inputs_path, prompts_path, SCHEMA_PATH, config_path)

    outputs_path = base / "outputs.jsonl"
    sim_config = SimulationConfig(error_rate=0.35, seed=7)
    write_outputs(
        simulate_records(load_inputs(inputs_path), sim_config),
        outputs_path,
    )

    final_dir = base / "final"
    run_transform(SCHEMA_PATH, outputs_path, final_dir, config_path)

    schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
    metrics = compute_metrics(str(outputs_path), str(GOLD_PATH), schema)
    return {
        "metrics": metrics,
        "outputs_path": outputs_path,
        "accepted_path": final_dir / "accepted.jsonl",
        "rejected_path": final_dir / "rejected.jsonl",
    }


baseline = run_config("baseline", BASELINE_CONFIG)
improved = run_config("improved", IMPROVED_CONFIG)
print("both runs complete")

both runs complete


## Raw model metrics (identical by construction)

`compute_metrics` reads the simulator outputs directly — those are the same bytes regardless of config, so the headline metrics match. Treat this as the *fixed model quality baseline*.

In [3]:
def row(name, b, i):
    delta = i - b
    arrow = "+" if delta >= 0 else ""
    print(f"{name:<32} {b:>8.2%}  {i:>8.2%}   {arrow}{delta:.2%}")


b, i = baseline["metrics"], improved["metrics"]
print(f"{'metric':<32} {'baseline':>8}  {'improved':>8}   delta")
print("-" * 70)
row("schema_validity_rate",            b.schema_validity_rate, i.schema_validity_rate)
row("completeness_rate",               b.completeness_rate, i.completeness_rate)
row("field_accuracy (overall)",        b.field_accuracy, i.field_accuracy)
row("consistency_rate",                b.consistency_rate, i.consistency_rate)
print(f"{'constraint_violations/record':<32} {b.constraint_violations_per_record:>8.2f}  {i.constraint_violations_per_record:>8.2f}")

metric                           baseline  improved   delta
----------------------------------------------------------------------
schema_validity_rate               66.67%    66.67%   +0.00%
completeness_rate                  97.22%    97.22%   +0.00%
field_accuracy (overall)           28.33%    28.33%   +0.00%
consistency_rate                  100.00%   100.00%   +0.00%
constraint_violations/record         0.17      0.17


## Transform-stage outcomes

Coercion targets *type* and *format* mismatches (e.g., stringifying a number, normalizing a date). It cannot recover `missing_required` or `invalid_enum` errors — those still reject. So the lift visible at this seed and error rate depends on which noise modes the simulator generated.

For this dataset (seed=7, error_rate=0.35) the simulator skews toward enum/required failures, so accepted counts are similar between configs. On data where type/format mismatches dominate, coercion typically lifts the accept rate by 5–15 points.

In [4]:
def count_lines(path):
    return sum(
        1 for line in path.read_text(encoding="utf-8").splitlines() if line.strip()
    )


b_acc = count_lines(baseline["accepted_path"])
b_rej = count_lines(baseline["rejected_path"])
i_acc = count_lines(improved["accepted_path"])
i_rej = count_lines(improved["rejected_path"])
total = b_acc + b_rej

print(f"{'stage':<14} {'baseline':>10} {'improved':>10}")
print("-" * 40)
print(f"{'accepted':<14} {b_acc:>10} {i_acc:>10}")
print(f"{'rejected':<14} {b_rej:>10} {i_rej:>10}")
print(f"{'accept rate':<14} {b_acc/total:>10.2%} {i_acc/total:>10.2%}")

stage            baseline   improved
----------------------------------------
accepted                8          8
rejected                4          4
accept rate        66.67%     66.67%


## Field accuracy breakdown

Per-field accuracy against the gold set. Reveals which semantic fields the model is getting wrong (often `category` or `priority`).

In [5]:
fields = sorted(set(b.field_accuracy_breakdown) | set(i.field_accuracy_breakdown))
print(f"{'field':<14} {'baseline':>10}  {'improved':>10}")
print("-" * 40)
for field in fields:
    bv = b.field_accuracy_breakdown.get(field, 0.0)
    iv = i.field_accuracy_breakdown.get(field, 0.0)
    print(f"{field:<14} {bv:>10.2%}  {iv:>10.2%}")

field            baseline    improved
----------------------------------------
category           33.33%      33.33%
customer_id         8.33%       8.33%
priority            0.00%       0.00%
summary             0.00%       0.00%
ticket_id         100.00%     100.00%


## Error taxonomy (top messages)

Counts of distinct schema-validation messages across the run. Long tail of one-off messages on baseline; the improved config should compress most of them.

In [6]:
def top_errors(metrics, n=5):
    return list(metrics.error_taxonomy.items())[:n]


print("BASELINE — top errors:")
for msg, count in top_errors(b):
    print(f"  {count:3d}  {msg[:100]}")

print("\nIMPROVED — top errors:")
for msg, count in top_errors(i):
    print(f"  {count:3d}  {msg[:100]}")

BASELINE — top errors:
    2  'critical' is not one of ['low', 'medium', 'high', 'urgent']
    2  'summary' is a required property

IMPROVED — top errors:
    2  'critical' is not one of ['low', 'medium', 'high', 'urgent']
    2  'summary' is a required property


## Sample rejection record

Each rejection in `rejected.jsonl` carries structured errors with `code`, `field_path`, `message`, and `raw_value` — that's what powers the cluster module in the next notebook.

In [7]:
rejected_lines = baseline["rejected_path"].read_text(encoding="utf-8").splitlines()
if rejected_lines:
    sample = json.loads(rejected_lines[0])
    sample.pop("record", None)  # focus on the errors
    sample.pop("raw", None)
    print(json.dumps(sample, indent=2)[:800])
else:
    print("no rejections (try a higher error rate)")

{
  "errors": [
    {
      "code": "enum",
      "field_path": "priority",
      "message": "'critical' is not one of ['low', 'medium', 'high', 'urgent']",
      "raw_value": "critical",
      "severity": "error"
    }
  ]
}


## Summary

The framework cleanly separates the *model layer* (deterministic simulator, identical by seed) from the *pipeline layer* (configurable parsing/coercion/prompting). On this dataset both paths land at the same accept rate because the simulator's dominant noise modes aren't coercible — but the comparison is now honest, reproducible, and the levers are config-only.

Continue with [03_monitoring_manifests_drift_clusters.ipynb](03_monitoring_manifests_drift_clusters.ipynb) to turn these reports into reproducible run records, drift checks, and clustered remediation hints.